This script uses climate data from climate_data_clean_v1.xlsx & stock price data of SP500 companies from SP500_daily_close_2000_2025.xlsx. 

Output: Df containing number of days taken for each company to recover (>= t-1 closing price of storm date) for each storm. 

In [56]:
import pandas as pd
import numpy as np

climate_file = "climate_data_clean_v1.xlsx"
climate_sheet = "storm_clean"

price_file = "SP500_daily_close_2000_2025.xlsx"

reshaped_price_file = "sp500_daily_close_long.xlsx"
output_file = "storm_company_recovery_days.xlsx"

start_date = pd.Timestamp("2015-01-01")

In [57]:
storm_df = pd.read_excel(climate_file, sheet_name=climate_sheet)

print(storm_df.head())
print(storm_df.columns)
print(storm_df.shape)

   disaster_subtype       event_name  start_date    end_date  \
0           Tornado  2000_jan_storm1    2000/1/2    2000/1/4   
1   Storm (General)  2000_jan_storm2   2000/1/22   2000/1/25   
2  Tropical cyclone           Leslie   2000/10/4   2000/10/4   
3   Storm (General)  2000_dec_storm1  2000/12/16  2000/12/17   
4           Tornado  2000_mar_storm1   2000/3/28   2000/3/28   

                                     affected_states  magnitude_kph  \
0                                           Kentucky            NaN   
1  Alabama, Georgia, Louisiana, Massachusetts, Ne...            NaN   
2                                            Florida            NaN   
3  Alabama, Connecticut, Florida, Georgia, Massac...            NaN   
4                                              Texas            NaN   

   total_damage_adjusted  
0                 382547  
1                 637579  
2                 398942  
3                 910827  
4                 819745  
Index(['disaster_subtype',

Cell 3 — keep needed storm columns and filter 

In [58]:
storm_df = storm_df[["event_name", "start_date"]].copy()
storm_df["start_date"] = pd.to_datetime(storm_df["start_date"], errors="coerce")

storm_df = storm_df.loc[storm_df["start_date"] >= start_date].copy()
storm_df = storm_df.sort_values("start_date").reset_index(drop=True)

print(storm_df.head())
print(storm_df.shape)
print("earliest storm date kept:", storm_df["start_date"].min())
print("latest storm date kept:", storm_df["start_date"].max())


        event_name start_date
0  2015_jan_storm1 2015-01-06
1  2015_jan_storm2 2015-01-26
2  2015_jan_storm3 2015-01-31
3          Octavia 2015-02-16
4  2015_feb_storm1 2015-02-16
(174, 2)
earliest storm date kept: 2015-01-06 00:00:00
latest storm date kept: 2025-12-08 00:00:00


Load stock price data

In [59]:
price_raw = pd.read_excel(price_file)

print(price_raw.head())
print(price_raw.columns[:10])  # first few columns only
print(price_raw.shape)

        Date  MOH      AZO  CMG      LRCX       ROP        DIS        KMB  \
0 2000-01-03  NaN  30.5625  NaN  3.745833  17.78125  29.471687  61.301533   
1 2000-01-04  NaN  30.4375  NaN  3.583333  16.96875  31.198063  60.762222   
2 2000-01-05  NaN  30.3125  NaN  3.531250  16.75000  32.492844  60.462608   
3 2000-01-06  NaN  29.0625  NaN  3.506250  16.75000  31.198063  61.361458   
4 2000-01-07  NaN  30.4375  NaN  3.558333  17.00000  30.704813  63.159157   

         WMB  MPC  ...        CL     SBAC       OKE        CMI  WYNN  \
0  23.198961  NaN  ...  31.12500  18.0000  5.430857  11.828125   NaN   
1  22.684469  NaN  ...  30.31250  17.7500  5.362458  11.500000   NaN   
2  24.087631  NaN  ...  29.28125  17.0000  5.417177  11.515625   NaN   
3  24.602125  NaN  ...  29.21875  16.9375  5.417177  11.921875   NaN   
4  25.256933  NaN  ...  30.96875  18.3750  5.622373  12.234375   NaN   

         KIM       SWKS  BG        L      SNA  
0  11.291667  31.406250 NaN  9.84375  26.8750  
1  11.18

reshape stock prices from wide to long and export for checking

In [60]:
price_raw = price_raw.rename(columns={price_raw.columns[0]: "date"})
price_raw["date"] = pd.to_datetime(price_raw["date"], errors="coerce")

# keep only rows from 2015-01-01 onward before reshaping
price_raw = price_raw.loc[price_raw["date"] >= start_date].copy()

price_long = price_raw.melt(
    id_vars="date",
    var_name="ticker",
    value_name="close"
)

price_long["close"] = pd.to_numeric(price_long["close"], errors="coerce")

# only drop missing dates, keep missing close values
price_long = price_long.dropna(subset=["date"]).copy()
price_long = price_long.sort_values(["ticker", "date"]).reset_index(drop=True)


print(price_long.head())
print(price_long.shape)
print("number of tickers:", price_long["ticker"].nunique())
print("earliest stock date kept:", price_long["date"].min())
print("latest stock date kept:", price_long["date"].max())


        date ticker      close
0 2015-01-02      A  40.560001
1 2015-01-05      A  39.799999
2 2015-01-06      A  39.180000
3 2015-01-07      A  39.700001
4 2015-01-08      A  40.889999
(1390795, 3)
number of tickers: 503
earliest stock date kept: 2015-01-02 00:00:00
latest stock date kept: 2025-12-30 00:00:00


helper functions

In [61]:
def get_pre_shock_info(company_prices, shock_date):
    prior_prices = company_prices.loc[company_prices["date"] < shock_date]

    if prior_prices.empty:
        return pd.NaT, np.nan

    last_row = prior_prices.iloc[-1]
    return last_row["date"], last_row["close"]


def get_recovery_info(company_prices, shock_date, pre_shock_price):
    if pd.isna(pre_shock_price):
        return pd.NaT, np.nan, np.nan

    future_prices = company_prices.loc[company_prices["date"] >= shock_date].copy()

    if future_prices.empty:
        return pd.NaT, np.nan, np.nan

    future_prices = future_prices.reset_index(drop=True)

    recovered = future_prices.loc[
        future_prices["close"].notna() & (future_prices["close"] >= pre_shock_price)
    ]

    if recovered.empty:
        return pd.NaT, np.nan, np.nan

    first_recovered_idx = recovered.index[0]
    first_recovered = future_prices.loc[first_recovered_idx]

    recovery_date = first_recovered["date"]
    recovery_price = first_recovered["close"]
    recovery_days = int(first_recovered_idx)

    return recovery_date, recovery_price, recovery_days

prepare ticker-level lookup

In [62]:
price_map = {
    ticker: grp.sort_values("date").reset_index(drop=True)
    for ticker, grp in price_long.groupby("ticker")
}

tickers = list(price_map.keys())

print("number of tickers:", len(tickers))
print("sample tickers:", tickers[:10])

number of tickers: 503
sample tickers: ['A', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACGL', 'ACN', 'ADBE', 'ADI', 'ADM']


compute recovery days for every storm × company

In [63]:
results = []

for _, storm in storm_df.iterrows():
    event_name = storm["event_name"]
    shock_date = storm["start_date"]

    for ticker in tickers:
        company_prices = price_map[ticker]

        pre_shock_date, pre_shock_price = get_pre_shock_info(company_prices, shock_date)
        recovery_date, recovery_price, recovery_days = get_recovery_info(
            company_prices,
            shock_date,
            pre_shock_price
        )

        # flag 1: whether a valid benchmark price exists
        benchmark_available = 0 if pd.isna(pre_shock_price) else 1

        # flag 2: whether the stock recovered by the end of the sample
        if benchmark_available == 0:
            recovered_by_end = np.nan
        elif pd.isna(recovery_days):
            recovered_by_end = 0
            recovery_days = -1
        else:
            recovered_by_end = 1

        results.append({
            "event_name": event_name,
            "shock_date": shock_date,
            "ticker": ticker,
            "pre_shock_date": pre_shock_date,
            "pre_shock_price": pre_shock_price,
            "recovery_date": recovery_date,
            "recovery_price": recovery_price,
            "recovery_days": recovery_days,
            "benchmark_available": benchmark_available,
            "recovered_by_end": recovered_by_end
        })

result_df = pd.DataFrame(results)

print(result_df.head())
print(result_df.shape)

        event_name shock_date ticker pre_shock_date  pre_shock_price  \
0  2015_jan_storm1 2015-01-06      A     2015-01-05        39.799999   
1  2015_jan_storm1 2015-01-06   AAPL     2015-01-05        26.562500   
2  2015_jan_storm1 2015-01-06   ABBV     2015-01-05        64.650002   
3  2015_jan_storm1 2015-01-06   ABNB     2015-01-05              NaN   
4  2015_jan_storm1 2015-01-06    ABT     2015-01-05        44.910000   

  recovery_date  recovery_price  recovery_days  benchmark_available  \
0    2015-01-08       40.889999            2.0                    1   
1    2015-01-06       26.565001            0.0                    1   
2    2015-01-07       66.930000            1.0                    1   
3           NaT             NaN            NaN                    0   
4    2015-01-08       45.680000            2.0                    1   

   recovered_by_end  
0               1.0  
1               1.0  
2               1.0  
3               NaN  
4               1.0  
(87522, 

validation checks

In [64]:
print(result_df["recovery_days"].describe())

print("\nmissing pre_shock_date:")
print(result_df["pre_shock_date"].isna().sum())

print("\nmissing pre_shock_price:")
print(result_df["pre_shock_price"].isna().sum())

print("\nmissing recovery_date:")
print(result_df["recovery_date"].isna().sum())

print("\nmissing recovery_price:")
print(result_df["recovery_price"].isna().sum())

print("\nmissing recovery_days:")
print(result_df["recovery_days"].isna().sum())

print("\nsample rows:")
print(result_df.sample(10, random_state=42))

count    84620.000000
mean        13.027641
std         71.096293
min         -1.000000
25%          0.000000
50%          0.000000
75%          3.000000
max       2642.000000
Name: recovery_days, dtype: float64

missing pre_shock_date:
0

missing pre_shock_price:
2902

missing recovery_date:
3487

missing recovery_price:
3487

missing recovery_days:
2902

sample rows:
            event_name shock_date ticker pre_shock_date  pre_shock_price  \
55117         Nicholas 2021-09-12     LW     2021-09-10        60.490002   
53990            Henri 2021-08-20     ES     2021-08-19        91.180000   
1246   2015_jan_storm3 2015-01-31    ICE     2015-01-30        41.146000   
54209            Henri 2021-08-20    RCL     2021-08-19        76.500000   
19452  2017_mar_storm1 2017-03-06   NTAP     2017-03-03        42.799999   
25133  2017_dec_storm1 2017-12-31     WM     2017-12-29        86.300003   
26701  2018_apr_storm1 2018-04-13   ARES     2018-04-12        21.400000   
77950  2024_may_stor

In [65]:
# 1. number of unique storms
num_storms = result_df["event_name"].nunique()
print("number of unique storms:", num_storms)

# 2. for each storm, count number of distinct tickers
ticker_count_by_storm = (
    result_df.groupby("event_name")["ticker"]
    .nunique()
    .reset_index(name="num_distinct_tickers")
    .sort_values("event_name")
    .reset_index(drop=True)
)

print(ticker_count_by_storm.head(20))
print(ticker_count_by_storm.shape)

print(ticker_count_by_storm["num_distinct_tickers"].describe())
print(ticker_count_by_storm["num_distinct_tickers"].value_counts().sort_index())

number of unique storms: 174
         event_name  num_distinct_tickers
0   2015_apr_storm1                   503
1   2015_apr_storm2                   503
2   2015_aug_storm1                   503
3   2015_dec_storm1                   503
4   2015_feb_storm1                   503
5   2015_jan_storm1                   503
6   2015_jan_storm2                   503
7   2015_jan_storm3                   503
8   2015_jul_storm1                   503
9   2015_jun_storm1                   503
10  2015_mar_storm1                   503
11  2015_mar_storm2                   503
12  2015_may_storm1                   503
13  2015_may_storm2                   503
14  2015_may_storm3                   503
15  2015_nov_storm1                   503
16  2015_oct_storm1                   503
17  2016_apr_storm1                   503
18  2016_apr_storm2                   503
19  2016_dec_storm1                   503
(174, 2)
count    174.0
mean     503.0
std        0.0
min      503.0
25%      503.0
50%  

export final output

In [66]:
result_df.to_excel(output_file, index=False, na_rep = "NA")
print("saved final output to:", output_file)

saved final output to: storm_company_recovery_days.xlsx
